# Concurrent multi-disease infection

Two independent diseases that can infect the same person at once: each disease
has its own status axis (`a`, `b`), its own force of infection, and its own
age-mixing matrix. Comorbidity shows up as:

- **susceptibility** — already infectious with B raises the rate of acquiring A
  (and vice versa) via `Multiply(..., where=...)` on the infection flow;
- **death** — co-infected people (`a=I & b=I`) have a higher exit rate.

Infection flows that need comorbidity multipliers are attached with `m.add_flow(TransitionFlow(...))`
and an explicit `ForceOfInfection`.

This page is summer4-native (not a summer2 port). See also
{doc}`08-strain-stratification` for exclusive multi-strain vs exclusive
multi-disease patterns.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    FlowModel,
    Compartments,
    ExitFlow,
    FlowMass,
    Multiply,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

# Independent SIR axes — the product allows concurrent infection (a=I & b=I).
status_a = Property("a", ("S", "I", "R"))
status_b = Property("b", ("S", "I", "R"))
age = Property("age", ("young", "old"))

pmap = (
    PropertyMap.from_property(status_a)
    .stratify(status_b)
    .stratify(age)
)
assert pmap.size == 18  # 3 × 3 × 2


def seed_y0() -> PropertyData:
    y = jnp.zeros(pmap.size)

    def put(sel, n: float) -> None:
        nonlocal y
        idx = np.asarray(pmap.select(sel))
        y = y.at[idx].set(n / len(idx))

    put(status_a["S"] & status_b["S"] & age["young"], 500.0)
    put(status_a["S"] & status_b["S"] & age["old"], 480.0)
    put(status_a["I"] & status_b["S"] & age["young"], 10.0)  # seed A in young
    put(status_a["S"] & status_b["I"] & age["old"], 10.0)  # seed B in old
    return PropertyData.wrap(pmap, y)


TS = jnp.linspace(0.0, 40.0, 81)
PLAN = SavePlan(
    requests={
        "comp": SaveRequest(Compartments()),
        "deaths": SaveRequest(FlowMass(flow="death")),
    },
    ts=TS,
)


## Separate mixing matrices

Disease A is strongly age-assortative; disease B mixes homogeneously. Each
`ForceOfInfection` carries its own `MixingMatrix` — they are not shared through
a per-FOI `MixingMatrix`.


In [ ]:
# Assortative contacts for disease A; homogeneous for disease B.
K_a = np.array([[0.9, 0.1], [0.1, 0.9]])
K_b = np.array([[0.5, 0.5], [0.5, 0.5]])
mix_a = MixingMatrix(age, K_a, normalize="rows", check_reciprocal=False)
mix_b = MixingMatrix(age, K_b, normalize="rows", check_reciprocal=False)

foi_a = ForceOfInfection(
    "inf_a",
    infectious=status_a["I"],
    group_by=age,
    mixing=mix_a,
    kind="frequency",
    contact_rate=Param("beta_a"),
)
foi_b = ForceOfInfection(
    "inf_b",
    infectious=status_b["I"],
    group_by=age,
    mixing=mix_b,
    kind="frequency",
    contact_rate=Param("beta_b"),
)

# Comorbid susceptibility: currently infectious with the *other* disease.
SUSCEPT_A_IF_B = 2.0
SUSCEPT_B_IF_A = 1.8
# Baseline infection death; co-infected multiply further.
DEATH_BASE = 0.01
DEATH_COMORBID = 5.0

m = FlowModel(pmap)
m.add_flow(
    TransitionFlow(
        "inf_a",
        status_a["S"],
        status_a["I"],
        foi_a,
        adjust=[Multiply(SUSCEPT_A_IF_B, where=status_b["I"])],
    )
)
m.add_flow(
    TransitionFlow(
        "inf_b",
        status_b["S"],
        status_b["I"],
        foi_b,
        adjust=[Multiply(SUSCEPT_B_IF_A, where=status_a["I"])],
    )
)
m.add_flow(TransitionFlow("rec_a", status_a["I"], status_a["R"], 0.2))
m.add_flow(TransitionFlow("rec_b", status_b["I"], status_b["R"], 0.25))
m.add_flow(
    ExitFlow(
        "death",
        status_a["I"] | status_b["I"],
        DEATH_BASE,
        adjust=[Multiply(DEATH_COMORBID, where=status_a["I"] & status_b["I"])],
    )
)

cm = m.compile()
# FOIs are carried on the infection TransitionFlow objects.
assert foi_a.mixing is mix_a and foi_b.mixing is mix_b
assert "inf_a" in cm.order and "inf_b" in cm.order
assert "inf_a" in cm.capture_meta and "inf_b" in cm.capture_meta


## Run and inspect comorbidity

Co-infected prevalence (`a=I & b=I`) should rise above the seed (zero). Death
mass should accumulate. Age curves for A should stay more assortative than B
(young A seed hits young harder under $K_a$).


In [ ]:
y0 = seed_y0()
params = {"beta_a": jnp.asarray(0.55), "beta_b": jnp.asarray(0.5)}
res = cm.run(params, y0, t0=0.0, t1=40.0, dt=0.1, save=PLAN, solver="euler")

both = res["comp"].select(status_a["I"] & status_b["I"]).total()
peak_both = float(jnp.max(jnp.asarray(both.values)))
assert peak_both > 1.0, f"expected concurrent infection, peak={peak_both}"

deaths = res["deaths"].integrate_intervals().cumulative().total()
final_deaths = float(jnp.asarray(deaths.values).ravel()[-1])
assert final_deaths > 0.0

# Early on, assortative Ka keeps more A infection in the seeded young band.
a_young = res["comp"].select(status_a["I"] & age["young"]).total()
a_old = res["comp"].select(status_a["I"] & age["old"]).total()
# Compare at day 5 (index 10 with TS linspace 0..40 / 81).
early = 10
ay = float(jnp.asarray(a_young.values).ravel()[early])
ao = float(jnp.asarray(a_old.values).ravel()[early])
assert ay > ao, f"early assortative A should favour young ({ay} vs {ao})"

frame = pd.DataFrame(
    {
        "A only": np.asarray(
            res["comp"].select(status_a["I"] & status_b["S"]).total().values
        ).ravel(),
        "B only": np.asarray(
            res["comp"].select(status_a["S"] & status_b["I"]).total().values
        ).ravel(),
        "A and B": np.asarray(both.values).ravel(),
    },
    index=np.asarray(TS),
)
frame.plot(
    title="Concurrent infection (totals)",
    labels={"index": "time", "value": "people"},
)


## Comorbid death adjustment is active

Compare cumulative deaths to a clone with the comorbidity multiplier removed.
The comorbid model must accumulate strictly more deaths from the same seed and
contact rates.


In [ ]:
m_base = FlowModel(pmap)
m_base.add_flow(TransitionFlow("inf_a", status_a["S"], status_a["I"], foi_a,
                               adjust=[Multiply(SUSCEPT_A_IF_B, where=status_b["I"])]))
m_base.add_flow(TransitionFlow("inf_b", status_b["S"], status_b["I"], foi_b,
                               adjust=[Multiply(SUSCEPT_B_IF_A, where=status_a["I"])]))
m_base.add_flow(TransitionFlow("rec_a", status_a["I"], status_a["R"], 0.2))
m_base.add_flow(TransitionFlow("rec_b", status_b["I"], status_b["R"], 0.25))
m_base.add_flow(ExitFlow("death", status_a["I"] | status_b["I"], DEATH_BASE))  # no comorbid multiply

res_base = m_base.compile().run(params, y0, t0=0.0, t1=40.0, dt=0.1, save=PLAN, solver="euler")
deaths_base = float(
    jnp.asarray(res_base["deaths"].integrate_intervals().cumulative().total().values).ravel()[-1]
)
assert final_deaths > deaths_base * 1.05, (
    f"comorbid death adjust should raise cumulative deaths "
    f"({final_deaths} vs {deaths_base})"
)
print(f"deaths with comorbidity multiplier: {final_deaths:.1f}")
print(f"deaths without:                    {deaths_base:.1f}")


## Comorbid susceptibility is active

Turn off the susceptibility multipliers but keep comorbid death. Peak
co-infection should fall.


In [ ]:
m_no_sus = FlowModel(pmap)
m_no_sus.add_flow(TransitionFlow("inf_a", status_a["S"], status_a["I"], foi_a))  # no suscept adjust
m_no_sus.add_flow(TransitionFlow("inf_b", status_b["S"], status_b["I"], foi_b))
m_no_sus.add_flow(TransitionFlow("rec_a", status_a["I"], status_a["R"], 0.2))
m_no_sus.add_flow(TransitionFlow("rec_b", status_b["I"], status_b["R"], 0.25))
m_no_sus.add_flow(
    ExitFlow(
        "death",
        status_a["I"] | status_b["I"],
        DEATH_BASE,
        adjust=[Multiply(DEATH_COMORBID, where=status_a["I"] & status_b["I"])],
    )
)
res_no_sus = m_no_sus.compile().run(params, y0, t0=0.0, t1=40.0, dt=0.1, save=PLAN, solver="euler")
peak_no_sus = float(
    jnp.max(
        jnp.asarray(
            res_no_sus["comp"].select(status_a["I"] & status_b["I"]).total().values
        )
    )
)
assert peak_both > peak_no_sus * 1.05, (
    f"comorbid susceptibility should raise peak co-infection "
    f"({peak_both} vs {peak_no_sus})"
)
print(f"peak co-infected with suscept adjust: {peak_both:.1f}")
print(f"peak co-infected without:             {peak_no_sus:.1f}")


## Under `jax.jit`

Differentiate through $\beta_A$ on the full comorbid model.


In [ ]:
def loss(beta_a):
    out = cm.run(
        {"beta_a": beta_a, "beta_b": params["beta_b"]},
        y0,
        t0=0.0,
        t1=40.0,
        dt=0.1,
        save=PLAN,
        solver="euler",
    )
    return jnp.sum(
        jnp.asarray(out["comp"].select(status_a["I"] & status_b["I"]).values.data)
    )


jitted = jax.jit(loss)
val = jitted(jnp.asarray(0.55))
grad = jax.grad(loss)(jnp.asarray(0.55))
assert jnp.isfinite(val) and jnp.isfinite(grad)
np.testing.assert_allclose(val, loss(jnp.asarray(0.55)), rtol=1e-4)
print(f"jit loss={float(val):.4g}, grad={float(grad):.4g}")


## Summary

| Piece | How |
|---|---|
| Concurrent infection | Product of per-disease status properties (`a` × `b`) |
| Separate mixing | One `MixingMatrix` per `ForceOfInfection` |
| Comorbid susceptibility | `Multiply` on infection `TransitionFlow` with `where=` other-disease `I` |
| Comorbid death | `Multiply` on `ExitFlow` with `where=a["I"] & b["I"]` |
| Infection with `adjust=` | `FlowModel.add_flow(TransitionFlow(..., ForceOfInfection(...)))` |

Cross-immunity or strain replacement would be extra transitions between these
compartments — not a different FOI primitive.
